In [1]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import shutil
import yaml
import torch
import csv
import random
from pathlib import Path
from ultralytics import YOLO

## Import the best model for the training

In [2]:
# Setup paths and load model
PROJECT_ROOT = Path(os.getcwd()).parent
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {results_dir}")

# Load best model - check multiple possible locations
possible_paths = [
    PROJECT_ROOT / 'runs/detect/yolov8n_vehicle_detection/weights/best.pt',
    PROJECT_ROOT / '../runs/detect/yolov8n_vehicle_detection/weights/best.pt',
]

model = None
for model_path in possible_paths:
    if model_path.exists():
        model = YOLO(str(model_path))
        print(f"✅ Trained model loaded from: {model_path}")
        break

# Fallback to base yolov8n model if trained model not found
if model is None:
    base_model_path = PROJECT_ROOT / 'yolov8n.pt'
    if base_model_path.exists():
        model = YOLO(str(base_model_path))
        print(f"⚠️  Using base model (not trained): {base_model_path}")
        print("Note: For better results, train the model using 'antoine train and test the model.ipynb'")
    else:
        print("❌ No model found. Please train the model first.")


Project root: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project
Results directory: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results
❌ No model found. Please train the model first.


## Compute flow and density metrics

In [ ]:
# Calculate flow and density

print(f"Device: {device}")

# Calculate Traffic Flow and Traffic Density using YOLOv8 built-in tracking
from collections import deque

class TrafficAnalyzer:
    """Analyze traffic flow and density using YOLOv8 tracking"""
    
    def __init__(self, counting_line_y=0.5, roi_start=0.3, roi_end=0.7, 
                 fps=25, road_length_meters=50):
        """
        Args:
            counting_line_y: Y-coordinate of counting line (normalized 0-1)
            roi_start: Start of ROI for density (normalized 0-1)
            roi_end: End of ROI for density (normalized 0-1)
            fps: Frames per second of video
            road_length_meters: Estimated road length in ROI (meters)
        """
        self.counting_line_y = counting_line_y
        self.roi_start = roi_start
        self.roi_end = roi_end
        self.fps = fps
        self.road_length_meters = road_length_meters
        
        # Track vehicles that crossed the counting line
        self.counted_tracks = set()  # Track IDs that crossed the line
        self.previous_positions = {}  # track_id: previous y-position
        
        # Metrics
        self.flow_count = 0
        self.density_history = deque(maxlen=1000)
        
    def update(self, results, frame_height):
        """
        Update with YOLOv8 track results
        results: YOLOv8 results object with boxes and track IDs
        frame_height: Height of frame in pixels
        """
        current_positions = {}
        
        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                # Get track ID (YOLOv8 assigns these automatically)
                if box.id is None:
                    continue
                
                track_id = int(box.id[0])
                
                # Get box center coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                y_center = (y1 + y2) / 2  # Center Y in pixels
                y_center_norm = y_center / frame_height  # Normalize to 0-1
                
                current_positions[track_id] = (y_center, y_center_norm)
                
                # Check if vehicle crossed the counting line
                if track_id in self.previous_positions:
                    prev_y_norm = self.previous_positions[track_id]
                    curr_y_norm = y_center_norm
                    
                    # Check for line crossing
                    if (prev_y_norm < self.counting_line_y <= curr_y_norm or 
                        prev_y_norm > self.counting_line_y >= curr_y_norm):
                        if track_id not in self.counted_tracks:
                            self.counted_tracks.add(track_id)
                            self.flow_count += 1
                
                # Update position
                self.previous_positions[track_id] = y_center_norm
        
        # Calculate density
        vehicles_in_roi = sum(1 for _, y_norm in current_positions.values()
                             if self.roi_start <= y_norm <= self.roi_end)
        density = vehicles_in_roi / self.road_length_meters if self.road_length_meters > 0 else 0
        self.density_history.append(density)
        
        return density
    
    def get_flow_rate(self, elapsed_minutes):
        """Get flow rate in vehicles per minute"""
        if elapsed_minutes > 0:
            return self.flow_count / elapsed_minutes
        return 0


def analyze_traffic_on_sequence(sequence_path, model, device, 
                                sample_rate=5, fps=25, road_length_m=50,
                                conf_threshold=0.25):
    """
    Analyze traffic flow and density on a video sequence using YOLOv8 tracking
    
    Args:
        sequence_path: Path to sequence folder
        model: YOLOv8 model
        device: 'cuda' or 'cpu'
        sample_rate: Process every Nth frame
        fps: Frames per second
        road_length_m: Road length in ROI (meters)
        conf_threshold: Detection confidence threshold
    """
    image_files = sorted(list(sequence_path.glob('*.jpg')) + 
                        list(sequence_path.glob('*.png')))
    
    if len(image_files) == 0:
        return None
    
    # Initialize tracker
    tracker = TrafficAnalyzer(
        counting_line_y=0.5,  # Middle of frame
        roi_start=0.3,
        roi_end=0.7,
        fps=fps,
        road_length_meters=road_length_m
    )
    
    sampled_images = image_files[::sample_rate]
    
    print(f"\nAnalyzing {sequence_path.name}...")
    print(f"Processing {len(sampled_images)} frames (sampled from {len(image_files)})")
    
    # Process all frames at once for proper tracking
    image_paths_str = [str(img_path) for img_path in sampled_images]
    
    for img_path in sampled_images:
        # Get image for frame height
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        frame_height = img.shape[0]
        
        # Use YOLOv8 built-in tracking - persist=True maintains IDs across calls
        results = model.track(
            source=str(img_path),
            conf=conf_threshold,
            device=device,
            verbose=False,
            persist=True  # Keep track IDs across frames
        )[0]
        
        # Update tracker
        tracker.update(results, frame_height)
    
    # Calculate metrics
    elapsed_time_minutes = (len(sampled_images) / fps) / 60
    flow_rate = tracker.get_flow_rate(elapsed_time_minutes)
    avg_density = np.mean(tracker.density_history) if tracker.density_history else 0
    max_density = np.max(tracker.density_history) if tracker.density_history else 0
    
    return {
        'sequence': sequence_path.name,
        'total_frames': len(image_files),
        'processed_frames': len(sampled_images),
        'vehicles_counted': tracker.flow_count,
        'flow_rate_veh_per_min': flow_rate,
        'avg_density_veh_per_m': avg_density,
        'max_density_veh_per_m': max_density,
        'elapsed_minutes': elapsed_time_minutes
    }


# Analyze traffic on TEST sequences
if model is not None:
    test_images_base = Path(PROJECT_ROOT) / "data_processed/test/images"
    
    # Get sequence folders
    test_sequences = sorted([d for d in test_images_base.iterdir() if d.is_dir()])
    
    if len(test_sequences) > 0:
        print(f"Found {len(test_sequences)} test sequences")
        
        # Analyze all test sequences
        num_sequences = len(test_sequences)
        traffic_results = []
        
        for seq_path in test_sequences:
            result = analyze_traffic_on_sequence(
                seq_path,
                model,
                device,
                sample_rate=10,  # Process every 10th frame
                fps=25,
                road_length_m=50,  # Estimate: 50 meters
                conf_threshold=0.25
            )
            if result:
                traffic_results.append(result)
        
        # Display results
        print("\n" + "="*100)
        print("TRAFFIC ANALYSIS RESULTS - TEST SET")
        print("="*100)
        
        for result in traffic_results:
            print(f"\n📊 Sequence: {result['sequence']}")
            print(f"   Duration: {result['elapsed_minutes']:.2f} minutes ({result['processed_frames']} frames)")
            print(f"   🚗 Vehicles Counted: {result['vehicles_counted']}")
            print(f"   📈 Flow Rate: {result['flow_rate_veh_per_min']:.2f} vehicles/minute")
            print(f"   🔢 Average Density: {result['avg_density_veh_per_m']:.3f} vehicles/meter")
            print(f"   📊 Max Density: {result['max_density_veh_per_m']:.3f} vehicles/meter")
        
        # Save to CSV
        if traffic_results:
            traffic_df = pd.DataFrame(traffic_results)
            results_dir.mkdir(parents=True, exist_ok=True)
            traffic_csv = results_dir / 'traffic_analysis_results_test.csv'
            traffic_df.to_csv(traffic_csv, index=False)
            print(f"\n✅ Results saved to: {traffic_csv}")
            
            # Summary statistics
            print("\n" + "="*100)
            print("OVERALL STATISTICS - TEST SET")
            print("="*100)
            print(f"Total sequences analyzed: {len(traffic_results)}")
            print(f"Average Flow Rate: {traffic_df['flow_rate_veh_per_min'].mean():.2f} vehicles/minute")
            print(f"Average Density: {traffic_df['avg_density_veh_per_m'].mean():.3f} vehicles/meter")
            print(f"Total vehicles counted: {traffic_df['vehicles_counted'].sum()}")
    else:
        print("❌ No test sequences found")
else:
    print("❌ Model not loaded")


Project root: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project
⚠️  Using base model (not trained): c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\yolov8n.pt
Note: For better results, train the model using 'antoine train and test the model.ipynb'
Device: cpu
Found 48 training sequences

Analyzing MVI_20012...
Processing 94 frames (sampled from 936)
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 25.9 MB/s  0:00:00

requirements: AutoUpdate success  2.0s
WARNING requirements: Restart runtime or rerun command for updates to take effect


Analyzing MVI_20032...
Processing 44 frames (sampled from 437)

Analyzing MVI_20034...
Processing 8